In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots

In [2]:
column_names = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety",
    "class"
]

cars = pd.read_csv("car.data", header=None, names=column_names)
cars

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


In [3]:
px.pie(cars, names="class")

In [4]:
px.histogram(
    cars[cars["doors"].isin(["2", "3", "4"])],
    x="safety",
    color="doors",
    barmode="group"
)

In [5]:
attributes = ["buying", "maint", "lug_boot", "safety", "class"]

cars_numeric = cars.copy()

category_num = {
    "low": 1,
    "med": 2,
    "high": 3,
    "vhigh": 4,
    "small": 1,
    "big": 3,
    "unacc": 1,
    "acc": 2,
    "good": 3,
    "vgood": 4
}

for attribute in attributes:
    cars_numeric[attribute] = cars_numeric[attribute].map(category_num)

# przykładowy samochód
car = cars_numeric.iloc[67]

fig = px.line_polar(
    r=[car[attr] for attr in attributes],
    theta=attributes,
    line_close=True,
    title="Wykres radarowy dla wybranego samochodu"
)

fig.update_traces(fill="toself")
fig.show()

In [6]:
my_choise = cars[
                ((cars["buying"] == "low") | (cars["buying"] == "med")) & 
                ((cars["maint"] == "low") | (cars["maint"] == "med")) & 
                ((cars["lug_boot"] == "med") | (cars["lug_boot"] == "big")) &
                (cars["safety"] == "high")
]

print("Ilość przefiltrowanych samochodów: ", len(my_choise))
px.histogram(my_choise, x="class")

Ilość przefiltrowanych samochodów:  96


In [7]:
cars["prediction"] = (
    ((cars["buying"] == "low") | (cars["buying"] == "med")) &
    ((cars["maint"] == "low") | (cars["maint"] == "med")) &
    ((cars["lug_boot"] == "med") | (cars["lug_boot"] == "big")) &
    (cars["safety"] == "high")
)

cars["actual"] = cars["class"].isin(["good", "vgood"])

correct = cars[cars["prediction"] == cars["actual"]]

print("Poprawne zgadnięcia:", len(correct))
print("Skuteczność:", len(correct) / len(cars) * 100, "%")

def calc_percentages(cars):
    prediction = cars["prediction"]
    actual = cars["actual"]

    correct_yes = (prediction & actual).sum()
    correct_no = (~prediction & ~actual).sum()
    incorrect_yes = (prediction & ~actual).sum()
    incorrect_no = (~prediction & actual).sum()

    n = len(cars)

    print(f"Poprawne TAK: {correct_yes / n * 100}%")
    print(f"Poprawne NIE: {correct_no / n * 100}%")
    print(f"Niepoprawne TAK: {incorrect_yes / n * 100}%")
    print(f"Niepoprawne NIE: {incorrect_no / n * 100}%")

    correct = correct_yes + correct_no
    print(f"Skuteczność: {correct / n * 100}%")


calc_percentages(cars)

Poprawne zgadnięcia: 1620
Skuteczność: 93.75 %
Poprawne TAK: 3.530092592592592%
Poprawne NIE: 90.2199074074074%
Niepoprawne TAK: 2.025462962962963%
Niepoprawne NIE: 4.224537037037037%
Skuteczność: 93.75%


Moja metoda skupia się na 4 atrybutach: cenie, kosztach utrzymania, pojemności bagaznika oraz bezpieczeństwie samochodu.
Samochód jest uwazany za dobry, jeśli:
- cena zakupu jest niska lub średnia (low lub med),
- koszty utrzymania są niskie lub średnie (low lub med),
- bagażnik jest średni lub duży (med lub big),
- bezpieczeństwo  jest wysokie (high).

Uznałam to za te wazniejsze cechy i nie uwzględniam inne, aby nie komplikować filtru. Skuteczność jest całkiem wysoka: ~93%.
Największym problemem reguły jest to, że niektóre samochody rzeczywiście dobre lub bardzo dobre zostają zaklasyfikowane jako niedobre.
Zróbmy szególowe wykresy dla takich samochodów.

In [8]:
missed_good_cars = cars[
    (cars["prediction"] == False) &
    (cars["class"].isin(["good", "vgood"]))
]

fig1 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Cena zakupu", "Koszt utrzymania"
    ]
)

fig2 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Pojemność bagażnika", "Bezpieczeństwo"
    ]
)

attributes = ["buying", "maint", "lug_boot", "safety"]

plot1 = px.histogram(
    missed_good_cars,
    x="buying",
    color="class",
    barmode="group",
    title="Ceny zakupu przeoczonych dobrych samochodów"
)

plot2 = px.histogram(
    missed_good_cars,
    x="maint",
    color="class",
    barmode="group",
    title="Koszty utrzymania przeoczonych dobrych samochodów"
)

for i, plot in enumerate([plot1, plot2]):
    for trace in plot.data:
        fig1.add_trace(trace, row=1, col=i+1)

fig1.show()

plot3 = px.histogram(
    missed_good_cars,
    x="lug_boot",
    color="class",
    barmode="group",
    title="Pojemność bagaznika przeoczonych dobrych samochodów"
)

plot4 = px.histogram(
    missed_good_cars,
    x="lug_boot",
    color="class",
    barmode="group",
    title="Bezpieczeństwo przeoczonych dobrych samochodów"
)

for i, plot in enumerate([plot3, plot4]):
    for trace in plot.data:
        fig2.add_trace(trace, row=1, col=i+1)

fig2.show()
# plot4.show()

In [12]:
good_cars = cars[((cars["class"] == "good") | (cars["class"] == "vgood")) 
                 & ((cars["buying"] == "high") | (cars["buying"] == "vhigh"))]
print(len(good_cars))

0


Na podstawie przeoczonych przypadków można wywnioskować, że część dobrych samochodów nie spełnia warunku dotyczącego 
wielkości bagażnika. Niektóre z nich mają mały bagażnik. Ponadto sporo dobrych samochodów ma wysokie koszty utrzymania,
przez co nie spełniają przyjętego kryterium.
Zaskoczeniem było dla mnie to, że 36 dobrych samochodów oraz 5 bardzo dobrych ma poziom bezpieczeństwa określony jako
"small" lub "med". Spodziewałam się, że w przypadku dobrych samochodów parametr ten będzie zawsze przyjmował najwyższą wartość.
Co ciekawe, w zbiorze nie występują dobre ani bardzo dobre samochody, których cena zakupu ma wartość "high" lub "vhigh".
Oznacza to, że warunek niskiej lub średniej ceny zakupu ("low" lub "med") jest bardzo skutecznym kryterium i warto zachować
go w przyjętej regule klasyfikacyjnej.